
# Table 1 Analysis

Compute Table 1 using integrated_dream_data.csv, treating each timepoint as a separate study (USA participants only). For each timepoint we report:
- N (unique participants)
- Age (median and population SD, age-bin midpoints)
- Gender: % Female, % Male, % Non-binary, % Other/Prefer not
- Education: % by education category

Runs on the standard library; pandas is optional for display.


In [31]:
from collections import Counter
import statistics
import csv

try:
    import pandas as pd  # optional; notebook works without it
except ImportError:
    pd = None

CSV_PATH = "integrated_dream_data.csv"

# Midpoints for age bins provided in the dataset
AGE_MIDPOINTS = {
    "18-24": 21,
    "25-34": 29.5,
    "35-44": 39.5,
    "45-54": 49.5,
    "55-64": 59.5,
    "65-74": 69.5,
    "75-84": 79.5,
    "85 years": 85,
}

print("Imports ready. Change CSV_PATH if your file is elsewhere.")

Imports ready. Change CSV_PATH if your file is elsewhere.


In [32]:
# Peek at columns and row count
with open(CSV_PATH, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)
print(f"Rows: {len(rows):,}")
print(f"Columns ({len(header)}): {header}")

Rows: 9,841
Columns (24): ['source_file', 'country', 'language', 'timepoint', 'participant_id', 'dream_text', 'dream_feelings', 'dream_talk', 'dream_write', 'dream_content', 'dream_frequency', 'dream_vivid', 'dream_bizarre', 'dream_emotional_tone', 'dream_intensity', 'gad_total', 'phq_total', 'age', 'education', 'student', 'gender', 'sleep_quality', 'sleep_hours', 'sleep_disturbed']


In [33]:
def classify_gender(raw):
    "Normalize gender strings/codes to coarse buckets."
    if raw is None:
        return None
    v = str(raw).strip().lower()
    if not v:
        return None
    if "female" in v:
        return "female"
    if "male" in v:
        return "male"
    if "non" in v and "binary" in v:
        return "non-binary"
    if "prefer" in v or "rather" in v:
        return "other"
    try:  # numeric codes sometimes appear
        code = int(float(v))
        if code == 1:
            return "female"
        if code == 2:
            return "male"
    except Exception:
        pass
    return "other"


def load_rows(path=CSV_PATH):
    "Return dataset as list of dict rows (stdlib only)."
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def detect_timepoints(rows):
    "Return sorted unique timepoints present among USA participants."
    tps = {
        row.get("timepoint")
        for row in rows
        if row.get("country") == "USA" and row.get("timepoint")
    }
    return sorted(tps, key=lambda x: (int(x) if str(x).isdigit() else x))


def detect_education_categories(rows):
    "Return sorted unique education categories among USA participants."
    cats = {
        (row.get("education") or "").strip()
        for row in rows
        if row.get("country") == "USA" and row.get("education")
    }
    # remove empty strings if any
    cats = {c for c in cats if c}
    return sorted(cats)


def compute_table(rows, timepoints=None, edu_cats=None):
    "Compute N, age median/SD, gender %, and education % per timepoint."
    if timepoints is None:
        timepoints = detect_timepoints(rows)
    if edu_cats is None:
        edu_cats = detect_education_categories(rows)
    results = {}
    for tp in timepoints:
        ids = set()
        ages = []
        genders = []
        edu_counts = Counter()
        for row in rows:
            if row.get("country") != "USA" or row.get("timepoint") != tp:
                continue
            pid = row.get("participant_id")
            if pid in ids:
                continue
            ids.add(pid)

            age_label = (row.get("age") or "").strip()
            if age_label in AGE_MIDPOINTS:
                ages.append(AGE_MIDPOINTS[age_label])

            gender = classify_gender(row.get("gender"))
            if gender:
                genders.append(gender)

            edu = (row.get("education") or "").strip()
            if edu:
                edu_counts[edu] += 1

        N = len(ids)
        median_age = statistics.median(ages) if ages else None
        sd_age = statistics.pstdev(ages) if len(ages) > 1 else None
        gcounts = Counter(genders)
        total_g = sum(gcounts.values())
        pct = lambda k: gcounts[k] / total_g * 100 if total_g else None

        total_edu = sum(edu_counts.values())
        edu_pct = {
            cat: (edu_counts[cat] / total_edu * 100 if total_edu else None)
            for cat in edu_cats
        }

        results[tp] = {
            "N": N,
            "median_age": median_age,
            "sd_age": sd_age,
            "female_pct": pct("female"),
            "male_pct": pct("male"),
            "nb_pct": pct("non-binary"),
            "other_pct": pct("other"),
            "edu_pct": edu_pct,
        }
    return results


rows = load_rows()
timepoints = detect_timepoints(rows)
edu_categories = detect_education_categories(rows)
print(f"Detected timepoints (treated as studies): {timepoints}")
print(f"Education categories: {edu_categories}")
table1 = compute_table(rows, timepoints, edu_categories)
table1

Detected timepoints (treated as studies): ['1', '2', '3', '4', '5']
Education categories: ['2 year degree', '4 year degree', 'Doctorate', 'High school graduate', 'Less than high school', 'Professional degree', 'Some college']


{'1': {'N': 505,
  'median_age': 49.5,
  'sd_age': 17.60881806036373,
  'female_pct': 51.68316831683168,
  'male_pct': 46.73267326732674,
  'nb_pct': 1.188118811881188,
  'other_pct': 0.39603960396039606,
  'edu_pct': {'2 year degree': 12.277227722772277,
   '4 year degree': 33.86138613861387,
   'Doctorate': 2.178217821782178,
   'High school graduate': 9.900990099009901,
   'Less than high school': 0.19801980198019803,
   'Professional degree': 20.198019801980198,
   'Some college': 21.386138613861387}},
 '2': {'N': 424,
  'median_age': 49.5,
  'sd_age': 17.413402432828068,
  'female_pct': 49.1725768321513,
  'male_pct': 49.40898345153664,
  'nb_pct': 1.1820330969267139,
  'other_pct': 0.2364066193853428,
  'edu_pct': {'2 year degree': 12.056737588652481,
   '4 year degree': 33.56973995271868,
   'Doctorate': 1.8912529550827424,
   'High school graduate': 10.638297872340425,
   'Less than high school': 0.2364066193853428,
   'Professional degree': 20.56737588652482,
   'Some college'

In [34]:
# Format for display; each timepoint is labeled Study 1, Study 2, ... in order


def fmt_age(rec):
    if rec["median_age"] is None or rec["sd_age"] is None:
        return ""
    return f"{rec['median_age']:.2f} ({rec['sd_age']:.2f})"


def fmt_pct(val):
    return f"{val:.2f}" if val is not None else ""


headers = ["Metric"]
for i, tp in enumerate(timepoints, start=1):
    headers.append(f"Study {i} (timepoint {tp}, N={table1[tp]['N']})")

rows_out = []
row_age = {"Metric": "Age (median, SD)"}
row_female = {"Metric": "Female (%)"}
row_male = {"Metric": "Male (%)"}
row_nb = {"Metric": "Non-binary (%)"}
row_other = {"Metric": "Other / Prefer not (%)"}

for i, tp in enumerate(timepoints, start=1):
    col = headers[i]
    rec = table1[tp]
    row_age[col] = fmt_age(rec)
    row_female[col] = fmt_pct(rec["female_pct"])
    row_male[col] = fmt_pct(rec["male_pct"])
    row_nb[col] = fmt_pct(rec["nb_pct"])
    row_other[col] = fmt_pct(rec["other_pct"])

rows_out.extend([row_age, row_female, row_male, row_nb, row_other])

# Education rows
for cat in edu_categories:
    r = {"Metric": f"Education: {cat}"}
    for i, tp in enumerate(timepoints, start=1):
        col = headers[i]
        r[col] = fmt_pct(table1[tp]["edu_pct"].get(cat))
    rows_out.append(r)

if pd is not None:
    display(pd.DataFrame(rows_out))
else:
    print(" | ".join(headers))
    print(" | ".join(["---"] * len(headers)))
    for r in rows_out:
        print(" | ".join(str(r.get(h, "")) for h in headers))

,Metric,"Study 1 (timepoint 1, N=505)","Study 2 (timepoint 2, N=424)","Study 3 (timepoint 3, N=383)","Study 4 (timepoint 4, N=348)","Study 5 (timepoint 5, N=320)"
0,"Age (median, SD)",49.50 (17.61),49.50 (17.41),49.50 (17.52),49.50 (17.66),49.50 (17.39)
1,Female (%),51.68,49.17,49.35,51.59,53.75
2,Male (%),46.73,49.41,49.35,47.84,45.94
3,Non-binary (%),1.19,1.18,1.31,0.58,0.31
4,Other / Prefer not (%),0.40,0.24,0.00,0.00,0.00
5,Education: 2 year degree,12.28,12.06,12.27,11.24,12.19
6,Education: 4 year degree,33.86,33.57,34.73,34.29,34.38
7,Education: Doctorate,2.18,1.89,1.83,2.02,1.56
8,Education: High school graduate,9.90,10.64,10.97,10.66,11.25
9,Education: Less than high school,0.20,0.24,0.26,0.29,0.31


In [35]:
# Persist to CSV for downstream use
import csv

out_path = "table1_summary.csv"
with open(out_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(rows_out)
print(f"Saved {out_path}")

Saved table1_summary.csv



Notes:
- All available timepoints among USA participants are treated as separate studies. Adjust by setting a custom `timepoints` list before calling `compute_table` if needed.
- Race/ethnicity fields are not in integrated_dream_data.csv, so only age, gender, and education are reported here.
- "Other" gender includes free-text responses that are neither male/female nor contain "non-binary", plus numeric codes outside 1/2 and any prefer-not statements.
- Education percentages are calculated over respondents with non-empty education data within each timepoint.
